# Group project: which cell types change in fly models of Parkinsonism?

**ISN Neurochemistry School 2026**

Parkinson's disease is defined by the loss of dopaminergic neurons. But patients
notice something else earlier: about a decade before any tremor, most of them
lose their sense of smell. Which cells are affected before the dying starts is
hard to study in people, because by the time a patient reaches a clinic it has
been going on for years.

Flies make it easy to look early. Pech et al. built five fly lines, each
carrying a mutation in the fly version of a human Parkinson's gene, and
sequenced every cell in the brains of young flies. For each cell type they
counted how many genes changed, and compared that with how many you would expect
given how many cells of that type they captured.

So the measurement is a count of changed genes per cell type, at one age. It
tells you where expression has shifted, which is not the same as where cells are
damaged or dying.

## Where this data comes from

**Pech et al. 2025**, "Synaptic deregulation of cholinergic projection neurons
causes olfactory dysfunction across five fly Parkinsonism models", *eLife*
98348. Read it at <https://elifesciences.org/articles/98348>.

- The three tables used below are the paper's **supplementary files 3, 5 and
  7**, listed on the article page under *Additional files*. You can download
  them by hand from there; the setup cell does it for you.
- The sequencing data is **GEO GSE228843**,
  <https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE228843>: 24 per-sample
  `.h5ad` files. They carry no cell type labels, which is why this project works
  from the authors' processed tables rather than from the raw matrices.
- A follow-up from the same group, **Ghezzi et al. 2026**, *eLife* 105386,
  reanalyses the same data and is worth reading if your result points at glia.

`scripts/download_data.py` prints every URL as it fetches, so you can always see
where a file came from.

## The five models

| model | human gene | what the protein normally does |
|---|---|---|
| `LRRK2G2019S` | *LRRK2* | a kinase; the commonest genetic cause of Parkinson's |
| `PINK1P399L` | *PINK1* | tags damaged mitochondria so the cell can clear them |
| `Rab39BG192R` | *RAB39B* | moves membrane cargo around the cell |
| `AuxR927G` | *DNAJC6*, auxilin | recycles vesicles at the synapse |
| `SynjR258Q` | *SYNJ1*, synaptojanin | recycles vesicles at the synapse |

Start with whichever interests you, then come back and run it again for another.
They do not all give the same answer, and the differences are worth more than
any single result.

**How this notebook works.** Loading and tidying the tables is done for you. The
analysis is not. Cells marked **Your turn** already run and print something, but
finishing the question takes a line or two that you write. Everything you need
is demonstrated in the cell just above.

Everything is `pandas` and `scanpy`. No model, no training.

## Running on Google Colab

Run the cell below first. On Colab it installs the software and fetches the
course files; on a laptop it does nothing. Run it either way.

In [ ]:
# Colab setup. On a laptop install this cell does nothing.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import os

    !pip install -q scanpy openpyxl "pandas==2.2.3"

    # Get the course files, or update them if this session already has them.
    import subprocess

    if os.path.isdir("/content/course/.git"):
        update = subprocess.run(
            ["git", "-C", "/content/course", "pull", "--quiet", "--ff-only"],
            capture_output=True, text=True,
        )
        if update.returncode:
            print("Could not update the course files. Continuing with the copy")
            print("already in this session, which is almost certainly fine.")
    else:
        !rm -rf /content/course
        !git clone -q https://github.com/nkempynck/isn-school-2026 /content/course
    %cd /content/course
    print("Ready.")
else:
    print("Running locally, nothing to install.")

Running locally, nothing to install.


## 0. Setup, and your model

Pick a model and put it in `MY_MODEL`. Everything below follows from it, so this
is the only line you have to change. Come back and change it to compare.

In [ ]:
MY_MODEL = "AuxR927G"     # <-- pick one of the five. Change this first.

MODELS = ["LRRK2G2019S", "PINK1P399L", "Rab39BG192R", "AuxR927G", "SynjR258Q"]
assert MY_MODEL in MODELS, f"pick one of {MODELS}"

import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.set_figure_params(figsize=(6, 5), dpi=90, frameon=False)
print("working on:", MY_MODEL)

working on: AuxR927G


## 1. The tables

Three supplementary files from the paper, loaded and tidied here so you can get
to the analysis. Published spreadsheets are laid out for human eyes rather than
for `pandas`, so this cell picks its columns by name instead of taking the
sheets as they come. Two things worth knowing if you ever load them yourself:
the cell type sheet has blank spacer columns and summary columns mixed into the
data, and in the gene sheet the gene names sit in a column `pandas` calls
`Unnamed: 10`, because the heading above that block spans several columns.

- `cells` : how far each cell type drifted from normal, per model
- `n_cells` : how many cells of each type were captured, summed over the mutants
- `genes` : every fly gene ranked by how deregulated it is, per model.
  **Rank 0 is the most deregulated**, so small numbers matter.

In [ ]:
from scripts.download_data import pech_tables

paths = pech_tables()

# how far each cell type drifted from normal
cells = pd.read_excel(paths["celltypes"], sheet_name="normalized residuals z_scores")
cells = (cells[["cell type"] + MODELS]
         .dropna(subset=["cell type"])
         .set_index("cell type")
         .apply(pd.to_numeric, errors="coerce")
         .dropna(how="all"))
cells.index = cells.index.astype(str).str.strip()

# how many cells of each type were captured. The count sheet spells the mutant
# names slightly differently, which is normal and worth noticing.
counts = pd.read_excel(paths["counts"], sheet_name="Cell Counts")
counts.index = counts["Unnamed: 0"].astype(str).str.strip()
MUTANT_COLS = ["AuxR927G", "hRab39BG192R", "SynjR258Q", "hPink1P399L", "hLRRK2G2019S"]
n_cells = counts[MUTANT_COLS].sum(axis=1)

# every gene, ranked per model
genes = pd.read_excel(paths["genes"], sheet_name="Ranked_Genes_Fly")
cols = ["Unnamed: 10"] + [m + ".1" for m in MODELS] + ["median rank"]
genes = genes[cols].dropna(subset=["Unnamed: 10"])
genes.columns = ["gene"] + MODELS + ["median_rank"]
genes = genes.set_index("gene").apply(pd.to_numeric, errors="coerce").dropna()

# clusters that were never annotated are left as numbers
named = cells[[not i.isdigit() for i in cells.index]]

print(f"\n{len(cells)} clusters, {len(named)} of them named")
print(f"{len(genes):,} genes ranked")
print(f"cell counts available for {len(set(cells.index) & set(n_cells.index))} clusters")

## 2. Which cell types change in your model?

One figure to start from. Large and positive means a cell type changed more than
its size predicts. Large and negative means it changed less, which happens to
the very big populations, so the plot shows the top twenty.

In [ ]:
mine = named[MY_MODEL].sort_values(ascending=False).head(20).sort_values()

fig, ax = plt.subplots(figsize=(7.2, 0.3 * len(mine) + 1.3))
ax.barh(range(len(mine)), mine.values,
        color=["#c0392b" if v > 0 else "#7d9bc1" for v in mine.values])
ax.set_yticks(range(len(mine)))
ax.set_yticklabels(mine.index, fontsize=9)
ax.axvline(0, color="#333333", lw=0.8)
ax.grid(False)
ax.set_xlabel("deregulation, further right means further from normal")
ax.set_title(f"{MY_MODEL}: the twenty cell types that drift furthest", fontsize=11)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
plt.show()

MY_TOP = named[MY_MODEL].idxmax()
print("top cell type:", MY_TOP)

Before going on, find out what that cell type is and what it does. If the
abbreviation means nothing to you, look it up. It is your whole result, and you
cannot say anything useful about it until you know what it is.

## 3. Your turn: is it just the number of cells?

Here is the first thing to worry about. The score is meant to be corrected for
how many cells of each type were captured, but corrections are never perfect,
and your top cell type might simply be a small cluster.

This is an ordinary check, the kind you would run on your own data. It usually
sharpens a result rather than weakening it, because it tells you which part of
the story will hold up.

In [ ]:
pair = pd.DataFrame({"score": cells[MY_MODEL], "n_cells": n_cells}).dropna()

print(f"{len(pair)} clusters have both a score and a cell count")
print("Spearman correlation:", round(pair.score.corr(pair.n_cells, method="spearman"), 2))
print("Pearson correlation: ", round(pair.score.corr(pair.n_cells), 2))

# YOUR TURN
# Plot score against n_cells so you can see the shape of it, and mark your own
# cell type. Use a log scale on x, because cluster sizes span three orders of
# magnitude. The pieces you need:
#
#     plt.scatter(pair.n_cells, pair.score, s=12)
#     plt.xscale("log")
#     plt.axhline(0)
#     plt.scatter(pair.loc[MY_TOP, "n_cells"], pair.loc[MY_TOP, "score"], color="red")
#     plt.xlabel(...); plt.ylabel(...); plt.show()
#
# Then answer: is the relationship strong enough to worry about?

## 4. Your turn: does your result survive the check?

The fair comparison is against clusters of a similar size, not against the whole
brain. Find the clusters within a factor of two of yours and see where you sit
among them.

In [ ]:
k = pair.loc[MY_TOP, "n_cells"]
band = pair[(pair.n_cells > k / 2) & (pair.n_cells < k * 2)]

print(f"{MY_TOP} was seen {int(k)} times across the five mutants.")
print(f"{len(band)} clusters are within a factor of two of that size.")
print(f"their median score is {band.score.median():.1f}, yours is {band.score[MY_TOP]:.1f}")

# YOUR TURN
# How many of those similar-sized clusters score higher than yours?
# Hint: a comparison like  band.score > band.score[MY_TOP]  gives True or False
# for each one, and .sum() counts the Trues.
#
# Then decide: does your cell type still stand out once size is accounted for?

## 5. How solid is it otherwise?

Two more quick checks on the evidence behind your number.

In [ ]:
# What does the authors' significance test add? They listed which cell types
# fall outside a confidence interval.
ci = pd.read_excel(paths["celltypes"], sheet_name="Cell types outside CIs").set_index("celltype")

if MY_TOP in ci.index:
    print(f"{MY_TOP} outside the confidence interval:")
    print(ci.loc[[MY_TOP]].to_string())
print(f"\nof {len(ci)} cell types, this many fall outside it in each model:")
print(ci.apply(lambda c: (c == True).sum()).to_string())

In [ ]:
# Do the three sheets describe the same clusters?
A = {i for i in cells.index if not i.isdigit()}
B = {str(i) for i in ci.index if not str(i).isdigit()}
C = {str(i) for i in counts.index if not str(i).isdigit()}

print(f"named clusters: scores {len(A)}, confidence intervals {len(B)}, counts {len(C)}")
print("in the score sheet but not in the counts:", sorted(A - C))

Worth knowing before you lean on either one. And one more thing to look up
yourself: on GEO, accession GSE228843, how many ages were sequenced, and for
which models? If you want to say one cell type is affected *before* another,
think about what that would take.

## 6. What about the dopaminergic neurons?

Parkinson's disease kills dopaminergic neurons. This brain has them, annotated
as central brain dopaminergic neurons, `CB DAN`. Compare them with your top cell
type, across all five models.

In [ ]:
cells.loc[["CB DAN", MY_TOP], MODELS].round(1)

## 7. Which genes change, and do the models agree?

The authors ranked every fly gene by how deregulated it is, separately for each
model. Rank 0 is the most deregulated.

In [ ]:
genes[MY_MODEL].nsmallest(15).astype(int)

Look several of those up. Do they have anything in common? Does it fit what your
disease gene normally does?

Be sceptical too. Ribosomal RNA, mitochondrial transcripts and histone genes
reach the top of a lot of rankings for technical reasons. If your list is full
of them, say so: that is a finding about the experiment.

In [ ]:
OTHER_MODEL = "PINK1P399L"      # pick a second model to compare against

top_mine = set(genes[MY_MODEL].nsmallest(100).index)
top_other = set(genes[OTHER_MODEL].nsmallest(100).index)

print(f"{MY_MODEL} and {OTHER_MODEL}")
print("genes in the top 100 of both:", len(top_mine & top_other))

# YOUR TURN
# 1. Which genes are those? Hint: sorted(top_mine & top_other)
# 2. Try a few different second models. Which pair agrees most, and does that
#    make sense from what the two proteins do?

## 8. Your turn: are those genes special to the affected cells?

If your model's genes were only made by the cell type that changed, that would
be a tidy explanation. Check it in the Fly Cell Atlas, the same file you used in
Practical 5.

In [ ]:
from scripts.prepare_data import make_fca_brain_h5ad

make_fca_brain_h5ad()          # 44 MB, from the course mirror
adata = sc.read_h5ad(ROOT / "data" / "processed" / "fca_brain.h5ad")

# The atlas names cell types by anatomy. Olfactory projection neurons are
# "antennal lobe projection neuron" here; find yours in the list if it differs.
TYPES = ["antennal lobe projection neuron", "dopaminergic PAM neuron",
         "gamma Kenyon cell", "columnar neuron T1", "ensheathing glial cell"]

MY_GENES = [g for g in genes[MY_MODEL].nsmallest(25).index if g in adata.var_names][:8]
print("showing:", MY_GENES)

sc.pl.dotplot(adata[adata.obs["annotation"].isin(TYPES)],
              var_names=MY_GENES, groupby="annotation")

# YOUR TURN
# Replace MY_GENES with genes you chose yourself and run it again. Then answer:
# are they specific to the affected cell type, or does every neuron have them?
# If every neuron has them, why would one cell type change first?
#
# To see which cell types the atlas has:  sorted(adata.obs["annotation"].unique())

## 9. Your results

Run this. It prints the headline together with what qualifies it.

In [ ]:
agree = named.corr(method="spearman")[MY_MODEL].drop(MY_MODEL).sort_values(ascending=False)

print(f"MODEL                 {MY_MODEL}")
print(f"most affected         {MY_TOP}  ({cells.loc[MY_TOP, MY_MODEL]:.0f})")
print(f"dopaminergic neurons  CB DAN  ({cells.loc['CB DAN', MY_MODEL]:.1f})")
print(f"top genes             {', '.join(genes[MY_MODEL].nsmallest(6).index)}")
print(f"agrees most with      {agree.index[0]}  (rho {agree.iloc[0]:.2f})")
print()
print(f"  built on            {int(k)} cells across the five mutants")
print(f"  score vs cell count Pearson {pair.score.corr(pair.n_cells):+.2f} "
      f"over {len(pair)} clusters")
print(f"  similar-sized       {len(band)} clusters within a factor of two")
print(f"  ages sequenced      one (day 5) for four of the five models")

## What to present

Three slides, about five minutes.

1. **Your gene and the experiment.** Which human Parkinson's gene you took, what
   the protein normally does, and what was measured: young flies, whole brain,
   before anything has degenerated.
2. **Your result.** The figure from section 2, your top cell type, and the
   dopaminergic number beside it. Then your genes and what they have in common.
3. **How far it goes.** The check from sections 3 and 4: that the score depends
   on cluster size, and whether your cell type still stands out once that is
   accounted for. Then the one experiment you would do next.

That third slide is what makes it a talk. Anyone can read the top of a sorted
table. Showing that you looked for the obvious problem with your own result, and
what you found, is the part people learn from.

If you looked at more than one model, say how they differed. If you looked at
one, say which and why.